*0.4 Deep learning basics*

# Seq2seq

**The situation.** Customers write dates every way there is: "March 3, 2024", "3/3/24", "3rd of March 2024". The database wants `2024-03-03`. Fifty regexes later, new formats still arrive. Turn the problem into a translation: input sequence of characters → output sequence of characters, learned from examples.

**Sequence-to-sequence.** Two recurrent networks. The *encoder* reads the input and compresses it into one vector (its final memory). The *decoder* starts from that vector and produces the output one token at a time, feeding each output back in as the next input. This is the 2014 design behind the first neural translation systems, and the shape of every generation model since.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The data.** 4,000 date pairs in several formats, character level. Made by the notebook, since the task is defined by its examples.

In [2]:
import random
from datetime import date, timedelta

import torch

random.seed(0)
MONTHS = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]


def random_pair():
    day = date(2000, 1, 1) + timedelta(days=random.randint(0, 365 * 30))
    formats = [
        f"{MONTHS[day.month - 1]} {day.day}, {day.year}",
        f"{day.day}/{day.month}/{day.year % 100:02d}",
        f"{day.day} {MONTHS[day.month - 1][:3]} {day.year}",
        f"{day.month:02d}-{day.day:02d}-{day.year}",
        f"{day.day}th of {MONTHS[day.month - 1]} {day.year}",
    ]
    return random.choice(formats), day.isoformat()


pairs = []
for _ in range(4000):
    pairs.append(random_pair())
print("examples:", pairs[:3])

all_text = ""
for source_text, target_text in pairs:
    all_text += source_text + target_text
alphabet = ["<pad>", "<start>", "<end>"] + sorted(set(all_text))
index = {}
for position, character in enumerate(alphabet):
    index[character] = position
PAD, START, END = 0, 1, 2


def encode(text, length):
    ids = []
    for character in text[:length]:
        ids.append(index[character])
    return ids + [PAD] * (length - len(ids))


IN_LEN, OUT_LEN = 24, 12
input_rows = []
target_rows = []
for source_text, target_text in pairs:
    input_rows.append(encode(source_text, IN_LEN))
    target_rows.append([START] + encode(target_text, OUT_LEN - 2) + [END])
inputs = torch.tensor(input_rows)
targets = torch.tensor(target_rows)
train_inputs, val_inputs = inputs[:3600], inputs[3600:]
train_targets, val_targets = targets[:3600], targets[3600:]
print(
    "alphabet:",
    len(alphabet),
    "characters | input",
    tuple(inputs.shape),
    "| target",
    tuple(targets.shape),
)

examples: [('04-12-2017', '2017-04-12'), ('25 Oct 2001', '2001-10-25'), ('12-07-2022', '2022-12-07')]
alphabet: 44 characters | input (4000, 24) | target (4000, 12)


**Encoder, decoder, and teacher forcing.** During training the decoder is fed the *correct* previous character (teacher forcing) so every position learns at once. At prediction time it feeds itself.

In [3]:
import time

import torch.nn.functional as F
from torch import nn


class Seq2Seq(nn.Module):
    def __init__(self, alphabet_size, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(alphabet_size, 32, padding_idx=PAD)
        self.encoder = nn.GRU(32, hidden, batch_first=True)
        self.decoder = nn.GRU(32, hidden, batch_first=True)
        self.output = nn.Linear(hidden, alphabet_size)

    def forward(self, source, target_in):
        _, memory = self.encoder(self.embedding(source))  # the whole input squeezed into one vector
        decoded, _ = self.decoder(
            self.embedding(target_in), memory
        )  # teacher forcing: true previous chars in
        return self.output(decoded)

    @torch.no_grad()
    def predict(self, source):
        _, memory = self.encoder(self.embedding(source))
        token = torch.full((source.shape[0], 1), START)
        out = []
        for _ in range(OUT_LEN - 1):
            decoded, memory = self.decoder(self.embedding(token), memory)
            token = self.output(decoded).argmax(dim=-1)  # feed our own output back in
            out.append(token)
        return torch.cat(out, dim=1)


def exact_match(model, source, target):
    model.eval()
    predicted = model.predict(source)
    return (predicted == target[:, 1:]).all(dim=1).float().mean().item()


def train(model, epochs=12, lr=3e-3, batch_size=64):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_inputs))
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            logits = model(train_inputs[batch], train_targets[batch][:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                train_targets[batch][:, 1:].reshape(-1),
                ignore_index=PAD,
            )
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if epoch % 3 == 0:
            print(
                
                    f"epoch {epoch:>2}  loss {loss.item():.3f}  exact-match on validation "
                    f"{exact_match(model, val_inputs, val_targets):.1%}"
                
            )
    return exact_match(model, val_inputs, val_targets)


torch.manual_seed(0)
started = time.perf_counter()
seq2seq = Seq2Seq(len(alphabet))
seq2seq_accuracy = train(seq2seq)
print(f"trained in {time.perf_counter() - started:.0f} s")


def decode(ids):
    text = ""
    for i in ids.tolist():
        if i == END:
            break
        text += alphabet[i]
    return text


for source in val_inputs[:3]:
    print(decode(source).strip("<pad>"), "→", decode(seq2seq.predict(source[None])[0]))
assert seq2seq_accuracy > 0.5

epoch  3  loss 0.860  exact-match on validation 0.0%


epoch  6  loss 0.631  exact-match on validation 0.5%


epoch  9  loss 0.394  exact-match on validation 11.2%


epoch 12  loss 0.147  exact-match on validation 62.7%
trained in 6 s
09-05-2015 → 2015-09-05
21 Nov 2013 → 2013-11-21
5 May 2025 → 2025-05-05


**Reading the output.** Exact-match accuracy climbs as the model learns the date grammar from examples; the samples show inputs in different formats mapped to ISO dates. Nobody wrote a format rule. Whatever it gets wrong is the bottleneck: the entire input has to fit through one 128-number vector.

```
"March 3, 2024" ──▶ encoder ──▶ [one vector] ──▶ decoder ──▶ 2 0 2 4 - 0 3 - 0 3
                                     ▲
                       everything the decoder will ever know
```

**The rule to remember.** Encoder compresses, decoder generates one token at a time from what it produced so far. The single-vector bottleneck is the reason attention (next item) was invented.

| Use it when | Don't when | Instead use |
|---|---|---|
| understanding where generation models came from; tiny normalisation tasks offline | anything you would ship today | a transformer, or an LLM with a schema for this exact task |

**Watch out**
- Teacher forcing at training vs self-feeding at prediction is a mismatch (*exposure bias*): one early mistake derails the rest.
- Greedy `argmax` decoding is the simplest; beam search was the standard upgrade.
- Long inputs collapse in the bottleneck; accuracy falls with input length — measure it.